# Model Comparison: Logistic Regression vs XGBoost
Compares the baseline and both XGBoost models on the same time-based split, and selects the final model using validation data.

In [1]:
%pip install xgboost imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    precision_score, recall_score, f1_score, confusion_matrix
)
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import os

RANDOM_STATE = 42
FP_COST = 2.0

# load the data


In [3]:
train_df = pd.read_csv("../data/train.csv")
val_df   = pd.read_csv("../data/validation.csv")
test_df  = pd.read_csv("../data/test.csv")

X_train, y_train = train_df.drop(columns=["Class"]), train_df["Class"].astype(int)
X_val,   y_val   = val_df.drop(columns=["Class"]),   val_df["Class"].astype(int)
X_test,  y_test  = test_df.drop(columns=["Class"]),  test_df["Class"].astype(int)

print("Train:", X_train.shape, "frauds:", y_train.sum())
print("Val:  ", X_val.shape,   "frauds:", y_val.sum())
print("Test: ", X_test.shape,  "frauds:", y_test.sum())

Train: (198608, 30) frauds: 366
Val:   (42559, 30) frauds: 55
Test:  (42559, 30) frauds: 52


# Training all models

In [4]:
# Ratio of normal to fraud cases, used by cost-sensitive XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_params = dict(
    objective="binary:logistic", eval_metric="aucpr",
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, n_jobs=-1
)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Cost-sensitive XGBoost": Pipeline([
        ("scaler", StandardScaler()),
        ("xgb", XGBClassifier(scale_pos_weight=scale_pos_weight, **xgb_params))
    ]),
    "SMOTE + XGBoost": ImbPipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
        ("xgb", XGBClassifier(scale_pos_weight=1.0, **xgb_params))
    ]),
}

val_probs, test_probs = {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    val_probs[name]  = model.predict_proba(X_val)[:, 1]
    test_probs[name] = model.predict_proba(X_test)[:, 1]
    print(f"{name}: trained ✓")

Logistic Regression: trained ✓
Cost-sensitive XGBoost: trained ✓
SMOTE + XGBoost: trained ✓


# Checking ranking quality

In [5]:
rows = []
for name in models:
    rows.append({
        "Model": name,
        "Val AP":  average_precision_score(y_val,  val_probs[name]),
        "Test AP": average_precision_score(y_test, test_probs[name]),
    })

ap_table = pd.DataFrame(rows).sort_values("Val AP", ascending=False).round(3)
ap_table

,Model,Val AP,Test AP
1,Cost-sensitive XGBoost,0.842,0.760
2,SMOTE + XGBoost,0.806,0.758
0,Logistic Regression,0.774,0.644


Cost-sensitive XGBoost performed best on validation and is selected; on test, both XGBoost approaches perform similarly.

# Chosing threshold for validation for each model

In [6]:
# choose a threshold for each model on validation

## Cost functions

In [7]:
def business_cost(y_true, preds, amounts):
    y_true, preds, amounts = np.asarray(y_true), np.asarray(preds), np.asarray(amounts, dtype=float)
    fp = (y_true == 0) & (preds == 1)
    fn = (y_true == 1) & (preds == 0)
    return {
        "FP": int(fp.sum()),
        "FN": int(fn.sum()),
        "total_cost": fp.sum() * FP_COST + amounts[fn].sum(),
    }

def best_threshold(y_true, probs, amounts):
    rows = []
    for t in np.arange(0.01, 1.00, 0.01):
        preds = (probs >= t).astype(int)
        rows.append({
            "threshold": round(t, 2),
            **business_cost(y_true, preds, amounts),
            "precision": precision_score(y_true, preds, zero_division=0),
            "recall": recall_score(y_true, preds, zero_division=0),
        })
    df = pd.DataFrame(rows).sort_values(["total_cost", "recall"], ascending=[True, False])
    return df.iloc[0]

## Finding the best model

In [8]:
#find the best threshold for each model:
val_amounts = val_df["Amount"].to_numpy()

thresholds = {}
rows = []
for name in models:
    best = best_threshold(y_val, val_probs[name], val_amounts)
    thresholds[name] = best["threshold"]
    rows.append({"Model": name, **best.to_dict()})

val_threshold_table = pd.DataFrame(rows).round(3)
val_threshold_table

,Model,threshold,FP,FN,total_cost,precision,recall
0,Logistic Regression,0.11,18.0,8.0,427.81,0.723,0.855
1,Cost-sensitive XGBoost,0.05,72.0,8.0,306.55,0.395,0.855
2,SMOTE + XGBoost,0.83,4.0,12.0,411.56,0.915,0.782


### Validation results: choosing a threshold for each model

I chose a threshold for each model on the validation data, picking the one with the lowest business cost (false alarm = $2, missed fraud = transaction amount). None of the best thresholds is 0.5, which shows that the default threshold does not suit this problem.

- **Cost-sensitive XGBoost** had the lowest cost ($306.55) at a threshold of 0.05. It caught 47 of the 55 frauds but raised 72 false alarms.
- **Logistic Regression** at a threshold of 0.11 missed the same number of frauds (8) with only 18 false alarms. Its cost was higher because the frauds it missed had larger transaction amounts.
- **SMOTE + XGBoost** was the most precise, with only 4 false alarms, but it missed more frauds (12).

Cost-sensitive XGBoost comes out cheapest mainly because a false alarm is assumed to cost only $2. If false alarms were more expensive, the ranking could change.

# Final evaluation on test data

In [9]:
# Get the transaction amount for every test transaction.
# A missed fraud costs its amount, so this is needed for the cost calculation.
test_amounts = test_df["Amount"].to_numpy()

# An empty list that will hold one row of results per model
rows = []

# Repeat the steps below for each of the three models
for name in models:

    # Use the threshold that was chosen on VALIDATION data.
    # It is not changed here; the test data is only for checking.
    t = thresholds[name]

    # Turn probabilities into 0/1 predictions:
    # probability >= threshold → 1 (fraud), otherwise → 0 (normal)
    preds = (test_probs[name] >= t).astype(int)

    # Save all the results for this model as one row
    rows.append({
        "Model": name,                       # model name
        "Threshold": t,                      # cutoff used

        # Ranking quality. Uses probabilities, so it does not depend on the threshold.
        "Test AP": average_precision_score(y_test, test_probs[name]),

        # Of the transactions flagged as fraud, how many really were fraud?
        "Precision": precision_score(y_test, preds, zero_division=0),

        # Of all the real frauds, how many did the model catch?
        "Recall": recall_score(y_test, preds, zero_division=0),

        # One score that balances precision and recall
        "F1": f1_score(y_test, preds, zero_division=0),

        # Count false alarms (FP) and missed frauds (FN), and work out the total cost:
        # FP × $2 + amounts of the missed frauds
        **business_cost(y_test, preds, test_amounts),
    })

# Turn the list of rows into a table, sort it so the cheapest model is on top,
# and round the numbers to 3 decimal places
final_table = pd.DataFrame(rows).sort_values("total_cost").round(3)

# Show the table
final_table

,Model,Threshold,Test AP,Precision,Recall,F1,FP,FN,total_cost
2,SMOTE + XGBoost,0.83,0.758,0.704,0.731,0.717,16,14,2405.16
1,Cost-sensitive XGBoost,0.05,0.760,0.342,0.750,0.470,75,13,2522.40
0,Logistic Regression,0.11,0.644,0.702,0.635,0.667,14,19,2772.47


### Final results on test data

I applied the thresholds chosen on validation to the test data (52 frauds) without changing them.

- **Logistic Regression** caught 33 frauds, missed 19, and raised 14 false alarms. It had the highest cost ($2,772).
- **Cost-sensitive XGBoost** caught 39 frauds, missed 13, and raised 75 false alarms (cost $2,522).
- **SMOTE + XGBoost** caught 38 frauds, missed 14, and raised only 16 false alarms. It had the lowest cost ($2,405).

Both XGBoost models clearly do better than Logistic Regression.

The two XGBoost models catch almost the same frauds, and the money lost from missed frauds is nearly identical (about $2,373 for each). The whole cost difference between them comes from false alarms (16 vs 75). With a false alarm at $2, SMOTE is cheaper by about $117. If a false alarm costs more, SMOTE becomes much better (for example, about $589 cheaper at $10 per false alarm).

Cost-sensitive XGBoost was chosen on validation, and I have not changed the choice based on test results. However, since the decision depends so strongly on the false-alarm cost, the final model should be confirmed with the DSA team's cost assumptions.

# Saving the  prediction outputs

In [ ]:

# Make sure the "results" folder exists (does nothing if it's already there)
os.makedirs("../results", exist_ok=True)

# Repeat for the validation and test data
for split, y_true, probs, df in [("validation", y_val,  val_probs,  val_df),
                                 ("test",       y_test, test_probs, test_df)]:

    # Start the table with the basic columns
    out = pd.DataFrame({
        "split": split,                          # which dataset this row is from
        "Amount": df["Amount"].values,           # transaction amount, for the cost analysis
        "actual_label": y_true.values,           # true answer: 1 = fraud, 0 = normal
    })

    # Add two columns per model: its probability and its 0/1 prediction
    for name in models:
        # Turn the model name into a simple column name,
        # e.g. "SMOTE + XGBoost" → "smote_xgboost"
        key = name.lower().replace(" + ", "_").replace("-", "_").replace(" ", "_")
        out[f"{key}_proba"] = probs[name]                                   # fraud probability
        out[f"{key}_pred"]  = (probs[name] >= thresholds[name]).astype(int) # 0/1 at the validation-chosen threshold

    # Save the table as a CSV file in the results folder
    out.to_csv(f"../results/predictions_{split}.csv", index=False)
    print(f"Saved predictions_{split}.csv  ({len(out)} rows)")

# Also save the chosen thresholds, so the DSA team knows which cutoff each model used
pd.Series(thresholds, name="threshold").to_csv("../results/model_thresholds.csv")
print("Saved model_thresholds.csv")

Saved predictions_validation.csv  (42559 rows)
Saved predictions_test.csv  (42559 rows)
Saved model_thresholds.csv
